# Assignment 02: Project AeroAssist

**Course:** Agentic AI: From Concepts to Practice

**Instructor:** Prof. Karthik Vaidhyanathan

**TAs:** Aviral Gupta, Aneetta Sara Shany, Ch Pavan Harshit, Shreyash Chandak

---

## Student Details

**Name:**

**Roll Number:**

**Email:**

---

## Instructions

- Run the notebook from top to bottom.
- Complete all coding tasks.
- Answer every reflection question in markdown.
- Do **not** hardcode your Gemini API key.
- Submit only the completed notebook.

# Part 1 — Building the Chatbot

## Objective

Create a Flask application that serves a simple chatbot interface.

By the end of this section you should have a working webpage with a message
box and a send button that talks to a Flask backend.

In [ ]:
# Install required packages
# Run this cell first — it may take a minute

!pip install flask flask-cors google-generativeai --quiet

In [ ]:
# Imports

import threading
import time
from flask import Flask, request, jsonify
from flask_cors import CORS
from google.colab.output import eval_js
import google.generativeai as genai

print('All libraries imported successfully.')

All libraries imported successfully.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Read your Gemini API key from Colab Secrets
#
# Steps:
#   1. Click the key icon in the left sidebar
#   2. Add a secret named GEMINI_API_KEY and paste your key as the value
#   3. Toggle 'Notebook access' ON

from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
print('API key loaded.' if GEMINI_API_KEY else 'WARNING: API key not found.')

API key loaded.


In [4]:
# Create the Flask application

app = Flask(__name__)
CORS(app)  # Allows the browser page to send requests to our backend

print('Flask app created.')

Flask app created.


In [5]:
# HTML for the chatbot interface
#
# This is a minimal starting point. The weird UI is deliberately made for you to fix.
# The three elements that matter are:
#   <div id='chat-window'>  — where messages will be displayed
#   <input id='user-input'> — where the user types
#   <button id='send-btn'>  — sends the message
#
# TODO: Build your HTML page below.
#       Add a <style> block inside <head> if you want to improve the look.
#       Hints: try background colours, font-family, padding, border-radius.

HTML_PAGE = """
<!DOCTYPE html>
    <html>
    <head>
        <title>Colab Chatbot</title>
        <style>
            body { font-family: Arial; background:#f4f4f4; display:flex; justify-content:center; }
            #chatbox { background:white; width:400px; margin-top:40px; padding:15px; border-radius:8px; }
            #messages { height:300px; overflow-y:auto; border:1px solid #ccc; padding:10px; }
        </style>
    </head>
    <body>
        <div id="chat-window">
            <h3>Chatbot</h3>
            <div id="messages"></div>
            <input id="user-input" placeholder="Type message..." />
            <button id="send-btn" onclick="sendMessage()">Send</button>
        </div>

"""

print('HTML template defined.')

HTML template defined.


In [6]:
# JavaScript that connects the frontend to the Flask backend
#
# The script below gives you the structure — fill in the two TODOs.
#
# How it works:
#   1. The user clicks Send (or presses Enter)
#   2. sendMessage() reads the text from #user-input
#   3. It sends a POST request to /chat with the message as JSON
#   4. The server replies with JSON: { response: '...', model: '...' }
#   5. addMessage() creates a new element and appends it to #chat-window

JS_SCRIPT = """
<script>

function addMessage(sender, text) {
    const chatWindow = document.getElementById("chat-window");
    chatWindow.innerHTML += `<div><b>${sender}:</b> ${text}</div>`;
}

function sendMessage() {

    const inputBox = document.getElementById("user-input");
    const message = inputBox.value;

    if (!message) return;

    addMessage("You", message);

    inputBox.value = "";

    fetch("/chat", {
        method: "POST",
        headers: {
            "Content-Type": "application/json"
        },
        body: JSON.stringify({
            message: message
        })
    })
    .then(response => response.json())
    .then(data => {
        addMessage("Bot", data.reply);
    });

}

</script>
</body>
</html>
"""

FULL_PAGE = HTML_PAGE + JS_SCRIPT
print("Page assembled.")

Page assembled.


In [7]:
url = eval_js('google.colab.kernel.proxyPort(5000)')
print('AeroAssist is running at:', url)
print('Open the link above in a new tab to test your chatbot.')

#
# The frontend reads data.response and displays it in the chat window.
# You do not need to modify this route — just replace the reply line in Part 2.

ACTIVE_MODEL = 'placeholder'

AeroAssist is running at: https://5000-m-s-kkb-usw4b0-ify9d1yv2hmi-b.us-west4-0.prod.colab.dev
Open the link above in a new tab to test your chatbot.


In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# Flask routes
#
# How the /chat endpoint works:
#
#   1. The browser sends:  POST /chat  with body { "message": "..." }
#   2. Flask reads the message from the request JSON
#   3. It generates a reply (placeholder for now; replaced in Part 2)
#   4. It returns:         { "response": "...", "model": "..." }

@app.route("/")
def home():
    return FULL_PAGE

@app.route('/chat', methods=['POST'])
def chat():
    message = request.json.get("message", "").lower()

    # Part 1: fixed reply so you can verify the frontend is working
    # TODO (Part 2): replace this with a call to askGemini(message)
    reply = f"[placeholder] You said: {message}"

    return jsonify({'reply': reply, 'model': ACTIVE_MODEL})


if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


Open the URL printed above in a new browser tab. Type a message and click Send. If you see the placeholder response appear in the chat window, Part 1 is complete.

> **Before moving on:** take a screenshot of the chatbot. Attach it in your submission zip.

# Part 2 — Connecting Gemini

## Objective

Replace the fixed placeholder reply with responses generated by Google's Gemini model.

By the end of this section your chatbot should answer real questions.

In [ ]:
# Configure the Gemini client

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite')

print('Gemini client configured.')

In [ ]:
# TODO: Complete the askGemini function.
#
# The function should:
#   - Accept a user prompt, an optional system prompt, and a temperature value
#   - Send the prompt to Gemini and return the response text
#
# The key call is:
  # response = gemini_model.generate_content(prompt, generation_config=config)
  # return response.text
#
# GenerationConfig controls temperature:
#   config = genai.types.GenerationConfig(temperature=temperature)
#
# If a system prompt is provided, prepend it to the user prompt before sending.

def askGemini(prompt, system=None, temperature=0.3):
    # your code here
    pass

# Test it
print(askGemini('What is the capital of France?'))

In [ ]:
# Update the /chat endpoint to use Gemini.
#
# Replace the placeholder reply with a call to askGemini(message).
# Re-run the cell below to relaunch the server with this change.

ACTIVE_MODEL = 'Gemini'

app.view_functions.pop('chat', None) # This is needed to remove the previously defined route. Use this wherever you redefine.
@app.route('/chat', methods=['POST'])
def chat():
    message = request.json.get('message', '')
    # TODO: replace this line with a call to askGemini(message)
    reply = f"[placeholder] You said: {message}"
    return jsonify({'reply': reply, 'model': ACTIVE_MODEL})

if __name__ == '__main__':
    app.run()

## Experiment

Ask your chatbot at least five different questions — a mix of general knowledge,
airline questions, and aircraft specifications.

Note anything interesting: does Gemini always answer the same way? Does it ever
make up airline-specific details it could not possibly know?


*(Write your observations here)*

# Part 3 — Understanding Your AI Assistant

## Objective

Investigate how different language models behave and how the way you instruct
a model shapes its responses.

This section is about experimentation — there are no single correct answers.

In [ ]:
import subprocess
import time

# Step 0: Install zstd, a compression utility required by Ollama
print('Installing zstd...')
!sudo apt-get install -y zstd

# Step 1: Download and install Ollama
print('Installing Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh

# Step 2: Start the Ollama server in the background
print('Starting Ollama server...')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for the server to be ready

# Step 3: Download a small language model
# gemma3:1b is a compact model from Google — about 815 MB. You are free to experiment with other models as well.
# This may take 2-4 minutes depending on your connection
print('Downloading gemma3:1b — this may take a few minutes...')
!ollama pull gemma3:1b

print('Done. Ollama is ready.')

In [ ]:
# TODO: Complete the askLocalModel function.
#
# Ollama exposes a local HTTP API at http://localhost:11434/api/generate
# Send a POST request with a JSON body and read the response.
#
# Required fields in the request body:
#   'model'  : the model name, e.g. 'gemma3:1b'
#   'prompt' : the user's message
#   'stream' : False  (so we get the full reply at once)
#
# Optional field:
#   'system' : a system prompt (same idea as in askGemini)
#
# The response JSON contains a 'response' key with the generated text.
#
# Hint: use the 'requests' library (import it as http_requests to avoid
#       clashing with Flask's 'request' object)

import requests as http_requests

OLLAMA_MODEL = 'gemma3:1b'

def askLocalModel(prompt, system=None, model=OLLAMA_MODEL, temperature=None):
    # your code here
    pass

# Test it
print(askLocalModel('What is the capital of France?'))

## Experiment 1 — Gemini vs Ollama

Use the five questions below to compare both models.
For each question, call both `askGemini` and `askLocalModel` and print the results side by side.

In [ ]:
# Five airline customer support questions
# You can swap any of these for your own if you prefer.

questions = [
    'What is the baggage allowance for economy class?',
    'Can I change my booking after purchasing my ticket?',
    'Does the Airbus A320 have Wi-Fi?',
    'How early should I arrive at the airport before my flight?',
    'What happens if my flight is cancelled?',
]

# TODO: Loop through the questions list.
#       For each question, call askGemini and askLocalModel.
#       Print the question and both responses clearly.


After running the comparison above, answer the following in a few sentences.

- How did the two models differ in speed and level of detail?
- Which gave more accurate or useful answers to the airline questions?
- Which would you rather put in front of a customer, and why?


## Experiment 2 — Temperature

Temperature controls how varied a model's output is.

- **0** — the model always picks the most likely next word (consistent, predictable)
- **0.5** — a balance between consistency and variety
- **1.0** — the model takes more creative risks (varied, sometimes unexpected)

Run the same prompt with all three values and compare the outputs.

In [ ]:
# TODO: Choose a customer support question and generate three responses
#       using temperatures 0, 0.5, and 1.0.
#       Print each temperature value and the corresponding response.

test_prompt = 'What should I do if I miss my flight?'

temperatures = [0, 0.5, 1.0]
# Hint: temperature=temp is all you need to set in the for loop

# your code here


## Experiment 3 — Prompting Strategies

The way you phrase a request to a language model has a large effect on the quality
of its response. In this experiment you will try three prompting strategies on the
same customer support task and compare the results.

Choose one task for all three sub-experiments. Good choices include:
- Classify a customer request (e.g. complaint / refund request / general enquiry)
- Summarise a customer complaint in one sentence
- Rewrite a draft reply in a professional airline tone


### (a) Zero-shot Prompting

Ask the model to perform the task without providing any examples.


In [ ]:
# Zero-shot: no examples, just the instruction and the input.

task_input = (
    "Customer message: 'I booked the wrong date and need to change my flight. "
    "The website is not letting me do it online. This is very frustrating.'"
)

zero_shot_prompt = f"""
Classify the customer request below into one of these categories:
Complaint, Booking Change, Refund Request, General Enquiry.
Then write a one-sentence professional reply.

{task_input}
"""

print('--- Zero-shot ---')
print(askGemini(zero_shot_prompt))


### (b) One-shot Prompting

Repeat the same task. This time, provide exactly one worked example before asking
your question.


In [ ]:
# One-shot: one example before the real task.

one_shot_prompt = f"""
Classify the customer request and write a one-sentence professional reply.

Example:
Customer message: 'My luggage was damaged during the flight.'
Category: Complaint
Reply: We sincerely apologise for the damage to your luggage and will arrange a claim review within 24 hours.

Now classify this:
{task_input}
Category:
Reply:
"""

print('--- One-shot ---')
print(askGemini(one_shot_prompt))


### (c) Few-shot Prompting

Repeat the experiment once more. Provide at least three examples before asking the
model to complete the same task.


In [ ]:
# Few-shot: three examples before the real task.

few_shot_prompt = f"""
Classify the customer request and write a one-sentence professional reply.

Example 1:
Customer message: 'My luggage was damaged during the flight.'
Category: Complaint
Reply: We sincerely apologise for the damage to your luggage and will arrange a claim review within 24 hours.

Example 2:
Customer message: 'Can I get a refund for my cancelled booking?'
Category: Refund Request
Reply: Yes, refunds for cancelled bookings are processed within 7 business days to your original payment method.

Example 3:
Customer message: 'Does AeroWing fly to Dubai?'
Category: General Enquiry
Reply: AeroWing currently operates flights to over 40 destinations; please check our website for the latest route map.

Now classify this:
{task_input}
Category:
Reply:
"""

print('--- Few-shot ---')
print(askGemini(few_shot_prompt))


Compare the three outputs above.

- Did the format of the response change as you added more examples?
- Did the accuracy or relevance of the category improve?
- What does this suggest about how many examples are worth providing?



*(Write your observations here)*

## Experiment 4 — System Prompts

A **system prompt** is an instruction you give the model before the conversation starts.
It defines the role, tone, and focus of every response.

Four example roles are provided below. Keep the user question fixed and observe
how each role changes the response.

In [ ]:
# Four role prompts for AeroWing staff
# Read through them — each one gives the model a different identity.

system_prompts = {
    'Customer Support Executive': (
        'You are a helpful and professional customer support executive at AeroWing Airlines. '
        'Answer questions clearly and concisely. If a question is outside your knowledge, '
        'say so politely and offer to escalate. Do not make up flight numbers, prices, or policies.'
    ),
    'Aircraft Maintenance Engineer': (
        'You are an aircraft maintenance engineer at AeroWing Airlines. '
        'Answer technical questions about aircraft systems, components, and safety procedures '
        'using precise engineering language. When in doubt, recommend consulting the AMM.'
    ),
    'Flight Operations Manager': (
        'You are a flight operations manager at AeroWing Airlines. '
        'You focus on scheduling, crew management, regulatory compliance, and operational efficiency. '
        'Answer questions from an operational planning perspective.'
    ),
    'Travel Consultant': (
        'You are a friendly travel consultant helping passengers plan their journeys with AeroWing Airlines. '
        'Focus on making travel enjoyable and stress-free.'
    ),
}

user_question = 'What should I know before boarding the aircraft?'

# TODO: Loop through system_prompts and call askGemini for each one,
#       passing the role's prompt as the 'system' argument.
#       Print the role name and the response for each.

# Hint: Iterate over all (role_name, system_prompt) pairs, call the model
# with the same user_question each time, and print both the role name
# and the generated response.

## Experiment 5 — Providing Additional Context

Gemini knows general aviation facts, but it does not know AeroWing's specific
aircraft or policies. One simple fix is to paste relevant information directly
into the prompt.

Compare responses with and without the AeroWing reference document below.

In [ ]:
# AeroWing quick-reference document
# This is the information the model does not have by default.

AEROWING_CONTEXT = """
AEROWING AIRLINES — AIRCRAFT QUICK REFERENCE

Airbus A320 (Narrow-body, Short-to-Medium Haul)
  Seating: 150-180 passengers (economy + business)
  Range: approximately 6,100 km
  Cruising speed: 840 km/h at 35,000 ft
  Wi-Fi: Available on select routes (check booking confirmation)
  Baggage: 23 kg checked + 7 kg cabin per Economy ticket
  Features: Winglets on A320neo variant for better fuel efficiency

Boeing 737-800 (Narrow-body, Short-to-Medium Haul)
  Seating: 160-190 passengers
  Range: approximately 5,765 km
  Cruising speed: 855 km/h
  Wi-Fi: Not available on this fleet
  Baggage: 23 kg checked + 7 kg cabin per Economy ticket

AeroWing Policies
  Booking changes: Allowed up to 24 hours before departure (fee may apply)
  Cancellations: Refundable within 24 hours of booking; non-refundable after
  Check-in: Online check-in opens 48 hours before departure
  Seat selection: Free for business class; fee applies for economy window/aisle
  Special meals: Request at least 48 hours before departure
"""

question = "Does AeroWing's A320 have Wi-Fi, and what is the baggage allowance?"

# TODO: Call askGemini twice with the same question:
#       1. Without any context — just pass the question as the prompt
#       2. With context — prepend AEROWING_CONTEXT to the prompt before
#          asking, and instruct the model to only use the provided information
#       Print both responses and compare.


## Reflection

1. Which experiment changed the model's responses the most — temperature, system prompts,
   or additional context? Why do you think that is?

2. What would happen if AeroWing had hundreds of pages of policy documents instead
   of a short paragraph? How do you think that problem could be solved?
   (You are not expected to implement a solution.)


(your answers here)

# Part 4 — Running a Local Language Model

## Objective

Modify the chatbot so that it can serve responses from either Gemini or Ollama,
without changing a single line of the frontend.

In [ ]:
# TODO: Write a unified ask() function that routes every request to
#       whichever model is currently selected by ACTIVE_MODEL.
#
# The function signature:
#   ask(prompt, system=None, temperature=0.3)
#
# If ACTIVE_MODEL == 'Gemini'  -> call askGemini(...)
# If ACTIVE_MODEL == 'Ollama'  -> call askLocalModel(...)
# Otherwise                    -> return '[No model selected]'
#
# The rest of the application will only ever call ask().
# This means switching models is a one-variable change.

def ask(prompt, system=None, temperature=0.3):
    # your code here
    pass

# Test it
ACTIVE_MODEL = 'Gemini'
print(ask('Say hello in one sentence.'))

In [ ]:
# TODO: Write a system prompt for AeroAssist.
#
# This prompt will be sent to whichever model is active.
# It should tell the assistant:
#   - What airline it works for
#   - What kind of questions it should answer
#   - How it should behave when it does not know something

CHATBOT_SYSTEM_PROMPT = (
    # your prompt here
    ''
)

print('System prompt defined.')

In [ ]:
# Model selector
#
# Change this variable to switch between Gemini and Ollama.
# Nothing else in the application needs to change.
#
# Options: 'Gemini'  or  'Ollama'

ACTIVE_MODEL = 'Gemini'   # <── change this to switch models

print(f'Active model: {ACTIVE_MODEL}')

In [ ]:
# Update the /chat route to use ask() and CHATBOT_SYSTEM_PROMPT.
# The /model route is provided — it lets the frontend show which model is active.

@app.route('/model')
def model_info():
    return jsonify({'model': ACTIVE_MODEL})

app.view_functions.pop('chat', None) # This is needed to remove the previously defined route. Use this wherever you redefine.
@app.route('/chat', methods=['POST'])
def chat():
    message = request.json.get('message', '')
    # TODO: replace this line with ask(message, system=CHATBOT_SYSTEM_PROMPT)
    reply = '[placeholder] replace me'
    return jsonify({'reply': reply, 'model': ACTIVE_MODEL})

if __name__ == '__main__':
    app.run()

> **Before moving on:** take a screenshot of the chatbot with Gemini active, then change `ACTIVE_MODEL = 'Ollama'`, re-run the selector cell, refresh the tab, and take a second screenshot. Attach both in your submission zip.

# Final Reflection

Answer each question in three to five sentences.
Support your answers with observations from your own experiments.

---

**1.** Compare cloud-hosted and locally hosted language models. Discuss one advantage and one limitation of each

---

**2.** Suppose AeroWing plans to deploy specialised assistants for Customer Support,
Flight Operations, and Aircraft Maintenance. Would you use the same system
prompt for every assistant or design a different one for each role? Explain your
reasoning.

---

**3.** Which of the experiments in Part 3 had the greatest influence on the quality of the
responses? Support your answer with observations from your own experiments.


(your answers here)